# **PLEASE SAVE A COPY OF THIS NOTEBOOK TO SUBMIT**

# Modeling: Multimodal AI - Homework 5
**MAS.S60 / 6.S985 - Spring 2026 - MIT**

# AI Agents in the Wild: Building, Evaluating, and Improving a Goal-Directed Agent

In this homework, you will design and implement an **AI agent** that operates in an environment, takes actions over multiple steps, and attempts to accomplish user-defined goals.

Unlike previous homeworks, this assignment is intentionally **open-ended**. Your team will choose an application domain and design an agent for it. The focus is not on building a polished product, but on building a **technically grounded agentic system** and rigorously analyzing its behavior.

You are encouraged to be ambitious and weird. However, your system must still satisfy the technical requirements below.

## Grading Overview
- Core homework total: **120 points**
- Optional extension: **up to +10 points** extra credit

---

## Learning Goals

By the end of this homework, you should be able to:

1. **Formulate an AI agent task as a sequential decision-making problem**
2. **Implement an agent loop** with observations, actions, state updates, and termination conditions
3. **Design and expose tools** for the agent through a structured interface
4. **Evaluate the behavior of your agent** on a benchmark of tasks
5. **Analyze failures** and identify which arise from model limitations vs. system design
6. **Improve the agent** through a technically motivated intervention
7. **Reflect on human oversight, safety, and the role of agency in interface design**

---

## Environment Setup

Go to the top menu:
Runtime → Change runtime type → Hardware accelerator → Choose **"A100"**

If you do not have Colab Pro, you can sign up for a free student Colab Pro account here:
https://colab.research.google.com/signup

# Part 1: Reading and Reflection (20 points)

#### Required Reading

Choose **3 papers/surveys** total:

##### Required core reading (pick at least 2)

1. A recent survey on multimodal LLM-based autonomous agents
2. A recent survey on agent optimization / training / post-training
3. A recent survey on agent evaluation / benchmarking

##### Domain-specific reading (pick at least 1)

Choose one area most relevant to your project:

- web agents
- GUI/computer-use agents
- social/simulated agents
- coding agents
- embodied / robotic agents
- human-agent interaction / human-in-the-loop systems
- other

##### Suggested papers/surveys (optional, non-exhaustive)

Use these as starting points for your 3 selected readings:

**Surveys**
- LLM Agent Methodologies and Applications (2025): https://arxiv.org/pdf/2503.21460
- Multimodal LLM Agent Methodologies (2025): https://arxiv.org/pdf/2510.10991
- LLM Agent Memory Engineering (2026): https://arxiv.org/html/2603.07670v1
- Optimization / Fine-tuning (2024): https://arxiv.org/html/2503.12434v2
- Planning (2024): https://arxiv.org/abs/2402.02716
- Building Effective Agents (2024): https://www.anthropic.com/research/building-effective-agents


**Key Papers**
- Toolformer (2023): https://arxiv.org/abs/2302.04761
- ReAct (2023): https://arxiv.org/abs/2210.03629
- MemGPT (2023): https://arxiv.org/abs/2310.08560
- SWE-agent (2024): https://arxiv.org/abs/2405.15793
- WebArena Benchmark (2023): https://arxiv.org/abs/2307.13854


---

## Questions

Based on your readings, answer the following in 1-2 paragraphs each.

### 1. What makes a system an **agent** rather than a chatbot or tool-using model?

Give a technical definition and describe the minimal ingredients required for agency.

### 2. Formalize your planned system as a **sequential decision problem**.

At minimum, define:

- observation space
- action space
- state / memory
- transition dynamics (informally is fine)
- objective or reward
- stopping condition

### 3. Compare two different agent architectures from the literature.

For example:

- ReAct vs planner-executor
- single-agent vs multi-agent
- direct tool use vs browser interaction
- static prompting vs reflection / self-critique
- prompting vs fine-tuning / RL-based improvement

### 4. What are the main evaluation challenges for your chosen kind of agent?

Be concrete. What counts as success? What metrics are misleading?

# Part 2: Observability and Evaluation Design (10 points)

Before building your agent, define how you will observe it and how you will evaluate it (agents that act in the world / outside the computer are also encouraged!). In agent systems, observability and evaluation are related but different: observability gives you traces, spans, and metrics about what happened; evaluation uses those signals to judge whether the behavior is good enough.

A useful mental model is that each run should be inspectable as a trace, with spans for key steps such as model calls and tool calls. This makes failures diagnosable: you can separate reasoning failures from tool failures, instruction failures, and infrastructure failures. Without this visibility, agents are black boxes and improvements become guesswork.

For this homework, we will start with offline evaluation before implementation is complete. Build a small but high-quality evaluation set (at least 10 tasks) with expected outcomes and a clear grading rule. Include normal cases, edge cases, and ambiguous/adversarial cases so your benchmark reflects realistic behavior rather than only easy prompts.

Define success in concrete terms. A strong definition includes final-answer correctness, trajectory quality (for example, whether required tools were used correctly), and operational quality (latency, cost, error rate). You should also specify which failures are critical versus acceptable tradeoffs.

Plan your observability schema now, then execute it later after implementation. At minimum, log trace ID, user query, per-step model outputs, tool calls, tool outputs, final answer, latency, cost/token usage, and a success label. In the final part, you will run the full evaluation loop after your implementation is complete.

Minimum requirements:
- Build an offline evaluation set with at least **10 tasks** and expected outcomes.
- Include at least **3 categories** of tasks: normal, edge, and ambiguous/adversarial.
- Define at least **3 metrics**: one correctness metric, one trajectory/process metric, and one operational metric (latency/cost/error).
- Specify a concrete grading rule for each metric (for example pass/fail threshold or score rubric).
- Propose a trace schema with required fields you will log in later parts.
- Document the above in your writeup.


Read more: [Agent Observability and Evaluation](https://huggingface.co/learn/agents-course/en/bonus-unit2/what-is-agent-observability-and-evaluation)

In [ ]:
# ============================================================
# ######################## CHANGE ME #########################
# ============================================================


# Example: load a small offline evaluation set before full implementation
!pip install datasets
from datasets import load_dataset

# You can replace this with your project-specific dataset (e.g., fact-checking).
DATASET_NAME = "openai/gsm8k"
DATASET_CONFIG = "main"
DATASET_SPLIT = "test[:20]"  # small slice for fast iteration

eval_dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split=DATASET_SPLIT)
print(f"Loaded {len(eval_dataset)} evaluation examples from {DATASET_NAME}/{DATASET_CONFIG}")
print("Columns:", eval_dataset.column_names)
print("Example:", eval_dataset[0])

# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

# Part 3: Build an Agent with smolagents (30 points)

In this part, you will implement a working agent in two stages:

1. **Stage A (existing tools):** build a baseline agent using built-in search/visit tools.
2. **Stage B (custom tools):** extend your agent with custom tools.

For this, we will use the open-source library [smolagents](https://huggingface.co/docs/smolagents/) which is a popular and versatile framework for building LLM agents.

#### Problem 1: GPU Verification and Library Installation

Run the following code cell to verify that your environment is correctly configured.

This step ensures that **PyTorch** and **CUDA** can access the GPU.
When the setup is correct, a **secret word** will appear in the output.

---

**In Your PDF Submission**

Include:
- A **screenshot** or **code snippet** showing the printed GPU information.
- The **secret word** displayed by your verification cell.

---

In [ ]:
!pip uninstall huggingface_hub -y
!pip install -q smolagents transformers accelerate bitsandbytes pillow torch torchvision trl peft datasets gdown qwen-vl-utils

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
t = torch.randn(2, 3, device=device)
KEY = 73
cipher_bytes = [8, 46, 44, 39, 61, 58, 105, 40, 59, 44, 105, 40, 42, 61, 32, 39, 46, 105, 60, 57]

if t.is_cuda:
    cipher = torch.tensor(cipher_bytes, dtype=torch.uint8, device=device)
    decoded = cipher ^ KEY
    secret_word = "".join(chr(c) for c in decoded.cpu().tolist())
    print(f"\nGPU check passed! Secret word: {secret_word}")
else:
    print("\nNo GPU detected. Please switch to an A100 runtime.")

#### Problem 2: Model & Libraries Setup (5 points)

Install dependencies and configure keys.

Requirements:

- Do **not** hardcode API keys in notebook code.
- Use environment variables.
- Record model name and tool list used for all experiments.

In [ ]:
import torch
from smolagents import TransformersModel

# ============================================================
# ######################## CHANGE ME #########################
# ============================================================

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required for local inference in this notebook.')

model = TransformersModel(
    model_id=MODEL_ID,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    max_new_tokens=1024,
    temperature=0.2,
    do_sample=False,
)
print('Local model configured on GPU:', MODEL_ID)

### Problem 3: Baseline Agent with Existing Tools (10 points)

Build a baseline tool-using agent with built-in tools first. See a complete list of built-in tools [here](https://huggingface.co/docs/smolagents/reference/default_tools). If you are feeling adventurous, you can also [use HuggingFace spaces](https://huggingface.co/docs/smolagents/reference/tools#smolagents.Tool.from_space) as tools.

Minimum requirements (also your deliverables):

- Use at least **two** built-in tools (for example, search plus webpage visit).
- Add a system instruction that defines scope and refusal behavior.
- Run at least **5** sample queries from your benchmark.
- Save raw outputs for each query.
- Report latency and success/failure label per query.

Short reflection (required):

- What did the agent do well?
- Where did it fail?
- Was the failure due to model reasoning, tool quality, or instruction design?

In [ ]:
# install packages required for websearch and page visit tools
!pip install -q markdownify requests

In [ ]:
from smolagents import CodeAgent, ToolCallingAgent, WebSearchTool, VisitWebpageTool
import datetime

SYSTEM_INSTRUCTIONS = (
    'You are a helpful agent for the MIT Modeling Multimodal AI 2026 course. '
    'Cite sources as URLs. If uncertain, say so explicitly.'
    'Course Website: https://mit-mi.github.io/mmai-course/spring2026/\n'
    f'Current date and time is. {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
)

baseline_agent = ToolCallingAgent(
    tools=[WebSearchTool(), VisitWebpageTool()],
    model=model
)

test_query = "What is a project one current student is working on? Find one and summarize it."
response = baseline_agent.run(f'{SYSTEM_INSTRUCTIONS}\n\nUser query: {test_query}')
print(response)

#### Problem 4: Custom Tool Integration (10 points)

Now extend your baseline with a custom tool and see if that changes the accuracy on your benchmark. This can be anything from integrating with an API such as Zillow or Google Drive to an image generator.

Minimum requirements (also your deliverables):

- Write at least **two** custom tools. Be creative.
- Re-run the same number of sample queries used in Problem 3.
- Save outputs for baseline and custom-tool versions side-by-side.
- Report at least one metric comparison (for example success rate, latency, or tool error rate).

Short reflection (required):

- What did the agent do, and did its performance improve? Why or why not?
- Where did it fail? Reflect on directions based on your readings on how to improve it (model choice, memory/state architecture, tools, etc.).

In [ ]:
from pathlib import Path
from smolagents import Tool

# ============================================================
# ######################## CHANGE ME #########################
# ============================================================

# download the latest student project info from the course repo
print("Downloading student project info file")
!curl -L "https://raw.githubusercontent.com/valleballe/mmai/master/static/student_repos.txt" -o student_repos.txt -q

class GetCurrentStudentProjectsTool(Tool):
    name = "get_current_student_projects"
    description = "Returns a string of all the students from 2026 and their project repository URLS for you to explore."
    inputs = {}
    output_type = "string"

    def forward(self) -> str:
        return Path("student_repos.txt").read_text(encoding="utf-8", errors="ignore")


# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

In [ ]:
from smolagents import CodeAgent, ToolCallingAgent, VisitWebpageTool, WebSearchTool
import datetime

course_agent = CodeAgent(
    tools=[GetCurrentStudentProjectsTool(), VisitWebpageTool(), WebSearchTool()],
    model=model,
)

SYSTEM_INSTRUCTIONS = (
    'You are a helpful agent for the MIT Modeling Multimodal AI 2026 course. '
    'Cite sources as URLs. If uncertain, say so explicitly. '
    'Course Website: https://mit-mi.github.io/mmai-course/spring2026/\n'
    f'Current date and time is {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
)

test_query = "What is a project one current student is working on? Find one and summarize it."
print(course_agent.run(f"{SYSTEM_INSTRUCTIONS}\n\nUser query: {test_query}"))

# Part 4: Build a Multimodal Language Agent (30 points)

Empowering agents with mutlimodal capabilities is crucial for solving tasks that go beyond text processing. For instance, many real-world challenges, such as web browsing, automatic purchasing, document understanding, or robotics, require analyzing rich visual content. Fortunately, smolagents provides built-in support for vision-language models (VLMs), enabling agents to process and interpret images effectively.

See architecture below:
https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/blog/smolagents-can-see/diagram_adding_vlms_smolagents.png

In this part, you will design and implement a multimodal language agent that solves a real multimodal task through multi-step interaction with an environment.

Your agent must use multimodal observations (such as images/screenshots/egocentric data, audio or other non-text modalities) as part of its decision-making, not only text prompts.

#### Problem 1: Vision Implementation and Controlled Comparison (20 points)

Build a vision-enhanced version of your previous agent.

Minimum requirements (also your deliverables):
- Start from your Part 3 agent and add multimodal observations into the decision loop. If multimodality is irrelevant to your task, for this exercise propose a new task (examples: web navigation, chart QA, UI automation, visual fact-checking, document understanding, image-based shopping assistant, map/screenshot reasoning, robot interaction).
- Implement an agent that consumes multimodal data at each step (e.g. image/screenshot input like in the template example below).
- Preserve step-level logs so we can inspect observation -> action -> result.
- Run a controlled comparison on the same task set:
  - Version A: text-only baseline
  - Version B: vision-enhanced agent (this section)

Required metrics:
- 1 architecture diagram
- Comparison of task success rate for Version A vs Version B
- 2 qualitative trace examples (one success, one failure)
- 1 short discussion of trade-offs
---

**NOTE: YOU MAY HAVE TO DOWNLOAD THIS NOTEBOOK AND RUN IT LOCALLY TO MAKE THE EXAMPLE CODE WORK**. The example code controls your chrome browser which google colab does not have.

In [ ]:
!pip install selenium helium pillow -q

**Action:** In this example, we will create a set of agent tools specifically designed for browsing, such as `search_item_ctrl_f`, `go_back`, and `close_popups`. These tools allow the agent to act like a person navigating the web. Note: It is possible to extend these with other navigation tools such as `mouse_click` and `keypress`.

In [ ]:
from io import BytesIO
from time import sleep

import helium
from PIL import Image
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

from smolagents import CodeAgent, Tool
from smolagents.agents import ActionStep


class SearchItemCtrlFTool(Tool):
    name = "search_item_ctrl_f"
    description = "Searches for text on the current page and jumps to the nth occurrence."
    inputs = {
        "text": {"type": "string", "description": "The text to search for"},
        "nth_result": {"type": "integer", "description": "Which occurrence to jump to", "nullable": True},
    }
    output_type = "string"

    def forward(self, text: str, nth_result: int = 1) -> str:
        driver = helium.get_driver()
        elements = driver.find_elements(By.XPATH, f"//*[contains(text(), '{text}')]")
        if nth_result > len(elements):
            raise Exception(f"Match n°{nth_result} not found (only {len(elements)} matches found)")
        result = f"Found {len(elements)} matches for '{text}'."
        elem = elements[nth_result - 1]
        driver.execute_script("arguments[0].scrollIntoView(true);", elem)
        result += f"Focused on element {nth_result} of {len(elements)}"
        return result

class GoBackTool(Tool):
    name = "go_back"
    description = "Goes back to previous page."
    inputs = {}
    output_type = "string"

    def forward(self) -> str:
        driver = helium.get_driver()
        driver.back()
        return "Went back to the previous page."

class ClosePopupsTool(Tool):
    name = "close_popups"
    description = "Closes visible modal or pop-up windows with ESC. This does not work on cookie consent banners."
    inputs = {}
    output_type = "string"

    def forward(self) -> str:
        driver = helium.get_driver()
        webdriver.ActionChains(driver).send_keys(Keys.ESCAPE).perform()
        return "Sent ESC to close popups (if any)."

**Perception:** In this example, we also need functionality for saving screenshots, as this will be an essential part of what our VLM agent uses to complete the task. This functionality captures the screenshot and saves it in step_log.observations_images = [image.copy()], allowing the agent to store and process the images dynamically as it navigates.

In [ ]:
from PIL import Image
import requests
from io import BytesIO
from time import sleep

import helium
from selenium import webdriver
from smolagents import ActionStep, CodeAgent

# Configure Chrome options
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--force-device-scale-factor=1")
chrome_options.add_argument("--window-size=1000,1350")
chrome_options.add_argument("--disable-pdf-viewer")
chrome_options.add_argument("--window-position=0,0")
# Added necessary options for Colab
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Initialize the browser with headless=True for Colab
driver = helium.start_chrome(headless=True, options=chrome_options)

def save_screenshot(step_log: ActionStep, agent: CodeAgent) -> None:
    sleep(1.0)  # Let JavaScript animations happen before taking the screenshot
    driver = helium.get_driver()
    current_step = step_log.step_number
    if driver is not None:
        prior_steps = getattr(getattr(agent, "memory", None), "steps", [])
        for step_logs in prior_steps:  # Remove previous screenshots from logs for lean processing
            if isinstance(step_logs, ActionStep) and step_logs.step_number <= current_step - 2:
                step_logs.observations_images = None
        png_bytes = driver.get_screenshot_as_png()
        image = Image.open(BytesIO(png_bytes))
        print(f"Captured a browser screenshot: {image.size} pixels")
        step_log.observations_images = [image.copy()]  # Create a copy to ensure it persists, important!

    # Update observations with current URL
    url_info = f"Current url: {driver.current_url}"
    step_log.observations = url_info if step_log.observations is None else step_log.observations + "\n" + url_info
    return

This function is passed to the agent as step_callback, as it’s triggered at the end of each step during the agent’s execution. This allows the agent to dynamically capture and store screenshots throughout its process.

Now, we can generate our vision agent for browsing the web, providing it with the tools we created. This tool will help the agent retrieve necessary information for verifying guests’ identities based on visual cues.

In [ ]:
helium_instructions = """
You can use helium to access websites. Don't bother about the helium driver, it's already managed.
We've already ran "from helium import *"
Then you can go to pages!
Code:
```py
go_to('github.com/trending')
```<end_code>

You can directly click clickable elements by inputting the text that appears on them.
Code:
```py
click("Top products")
```<end_code>

If it's a link:
Code:
```py
click(Link("Top products"))
```<end_code>

If you try to interact with an element and it's not found, you'll get a LookupError.
In general stop your action after each button click to see what happens on your screenshot.
Never try to login in a page.

To scroll up or down, use scroll_down or scroll_up with as an argument the number of pixels to scroll from.
Code:
```py
scroll_down(num_pixels=1200) # This will scroll one viewport down
```<end_code>

When you have pop-ups with a cross icon to close, don't try to click the close icon by finding its element or targeting an 'X' element (this most often fails).
Just use your built-in tool `close_popups` to close them:
Code:
```py
close_popups()
```<end_code>

You can use .exists() to check for the existence of an element. For example:
Code:
```py
if Text('Accept cookies?').exists():
    click('I accept')
```<end_code>

# IMPORTANT
When outputting code output it in <code> tags.

# ONLY USE AVAILABLE TOOLS!
"""

In [ ]:
import os

from smolagents import CodeAgent, OpenAIServerModel

# ============================================================
# ############ IF YOU CANT RUN THE MODEL LOCALLY #############
# ============================================================
if os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY").strip()

model = OpenAIServerModel(model_id="gpt-5.4-mini")
# ============================================================
# ########## END IF YOU CANT RUN THE MODEL LOCALLY ###########
# ============================================================

# Create the agent
agent = CodeAgent(
    tools=[GoBackTool(), ClosePopupsTool(), SearchItemCtrlFTool()],
    model=model,
    additional_authorized_imports=["helium"],
    step_callbacks=[save_screenshot],
    max_steps=20,
    verbosity_level=2,
)

# Import helium for the agent
agent.python_executor("from helium import *")

In [ ]:
search_request = """
Please navigate to https://en.wikipedia.org/wiki/Chicago and give me a sentence containing the word "1992" that mentions a construction accident.
"""

agent_output = agent.run(search_request + helium_instructions)
print("Final output:")
print(agent_output)

#### Problem 2: Safety and Policy Evaluation (10 points)

Add a targeted safety/policy evaluation for your agent and compare behavior before and after one mitigation.

Minimum requirements (also your deliverables):
- Design at least **3** challenging prompts relevant to your domain (for example: unsafe requests, privacy-sensitive requests, or out-of-scope requests).
- Define expected safe behavior for each prompt before running the test.
- Run your agent on all prompts and record observed behavior.
- Implement at least one mitigation (for example: improved system instruction, tool guardrail, or explicit refusal policy) and re-run the same prompts.
- Report one table with **before/after** behavior and a short reflection on trade-offs (false refusals vs missed refusals).

In [ ]:
# ============================================================
# ######################## CHANGE ME #########################
# ============================================================

from smolagents import CodeAgent, VisitWebpageTool, WebSearchTool
import datetime


SYSTEM_INSTRUCTIONS = (
    'You are a helpful agent for the MIT Modeling Multimodal AI 2026 course. '
    'Cite sources as URLs. If uncertain, say so explicitly.'
    'Course Website: https://mit-mi.github.io/mmai-course/spring2026/\n'
    f'Current date and time is. {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
)

test_query = "What is a project one current student is working on? Find one and summarize it."
print(agent.run(f"{SYSTEM_INSTRUCTIONS}\n\nUser query: {test_query}"))

# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

# Part 5: Agent Observability and Evaluation (20 points)

So far, we have done offline evaluation of our model on a subset of our dataset with ground truth labels. In this part, we will explore observability and online evaluation of our agent. In the following demonstration, we will use Langfuse as our observability tool, but you can use any other OpenTelemetry-compatible services (like TruLens). The code below shows how to set environment variables for Langfuse (or any OTel endpoint) and how to instrument your smolagent.

#### Problem 1: Setting up an Observability Monitor (5 points)

Set up an observability backend (Langfuse or any OpenTelemetry-compatible service) so each agent run is traceable end-to-end.

In [ ]:
# STEP 1: Install dependencies
!pip install -q langfuse 'smolagents[telemetry]' openinference-instrumentation-smolagents datasets 'smolagents[gradio]' gradio --upgrade

In [ ]:
import os
from getpass import getpass

# Get keys for your project from the project settings page: https://us.cloud.langfuse.com
# Use environment variables only; do not hardcode secrets in notebook source.
if not os.getenv("LANGFUSE_SECRET_KEY"):
    os.environ["LANGFUSE_SECRET_KEY"] = getpass("Enter LANGFUSE_SECRET_KEY: ")

if not os.getenv("LANGFUSE_PUBLIC_KEY"):
    os.environ["LANGFUSE_PUBLIC_KEY"] = getpass("Enter LANGFUSE_PUBLIC_KEY: ")

# You can override this if you use self-hosted Langfuse.
os.environ.setdefault("LANGFUSE_BASE_URL", "https://us.cloud.langfuse.com")

print("Langfuse environment variables configured.")

## Remember to remove the keys for before your submission!


In [ ]:
from langfuse import get_client

langfuse = get_client()

# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

#### Problem 2: Record and Inspect Traces (5 points)

Next, set up SmolagentsInstrumentor() to instrument your smolagent and send traces to Langfuse. Then run your Part 3/4 agent and inspect trace behavior in the dashboard.

Minimum requirements:
- Show evidence that at least 5 runs were recorded as traces.
- For at least 2 runs, inspect spans and identify: model call count, tool call sequence, and where most latency occurred.
- Include at least 1 run where behavior was incorrect or suboptimal, and diagnose whether the issue came from reasoning, tool output, prompt/instructions, or infrastructure.
- Link each diagnosis to specific trace evidence (span names, timing, tool output, or error text).

In [ ]:
!pip install markdownify requests

In [ ]:
from openinference.instrumentation.smolagents import SmolagentsInstrumentor
import datetime
SmolagentsInstrumentor().instrument()

# ============================================================
# ######################## CHANGE ME #########################
# ============================================================

from smolagents import CodeAgent, VisitWebpageTool, WebSearchTool

course_agent = CodeAgent(
    tools=[VisitWebpageTool(), WebSearchTool()],
    model=model,
)

SYSTEM_INSTRUCTIONS = (
    'You are a helpful agent for the MIT Modeling Multimodal AI 2026 course. '
    'Cite sources as URLs. If uncertain, say so explicitly.'
    'Course Website: https://mit-mi.github.io/mmai-course/spring2026/\n'
    f'Current date and time is. {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
)

test_query = "What are the current student projects?"
print(course_agent.run(f"{SYSTEM_INSTRUCTIONS}\n\nUser query: {test_query}"))

# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

Check your [Langfuse Traces Dashboard](https://cloud.langfuse.com/) (or your chosen observability tool) to confirm that the spans and logs have been recorded.

#### Problem 3: Online Evaluation (10 points)

In a previous section, we learned about the difference between observability, online, and offline evaluation. Now, we will monitor your agent under live-like conditions and evaluate trade-offs across configuration choices.

Read more: [Monitoring and evaluating agents](https://huggingface.co/learn/agents-course/en/bonus-unit2/monitoring-and-evaluating-agents-notebook).

Common metrics include:
- Costs: token usage, which you can transform into approximate costs by assigning a price per token.
- Latency: time it takes to complete each step, or the entire run.
- User feedback: in real-life deployment, users can often provide direct feedback to help refine or correct the agent (such as thumbs up or down with explanation).
- LLM-as-a-judge: use a separate LLM to evaluate your agent's output in near real-time (e.g., checking for toxicity, correct tool use, user response quality, or correctness).

Minimum requirements:
- Change at least two parameters of your agent such as the LLM model, planning steps, tool set size, or memory architecture (for inspiration see the [smolagents documentation](https://huggingface.co/docs/smolagents/)).
- Evaluate each configuration on the same set of at least 5 prompts.
- Track at least 3 metrics per configuration (for example success rate, average latency, and estimated cost).
- Attach screenshots of relevant Langfuse results in your hand-in.

Deliverables:
- One comparison table with each configuration and all reported metrics.
- A short discussion (6-8 sentences): how your parameter changes impacted results, where trade-offs appeared, and which setup you would deploy. Consider how user feedback or LLM-as-a-judge could be integrated in future online evaluations.

In [ ]:
# You can make write your agents with different architectural changes here:


# ============================================================
# ######################## CHANGE ME #########################
# ============================================================

from smolagents import CodeAgent, VisitWebpageTool, WebSearchTool

course_agent = CodeAgent(
    tools=[GetCurrentStudentProjectsTool(), VisitWebpageTool(), WebSearchTool()],
    model=model,
)

SYSTEM_INSTRUCTIONS = (
    'You are a helpful agent for the MIT Modeling Multimodal AI course. '
    'Cite sources as URLs. If uncertain, say so explicitly.'
)

test_query = "What are the current student projects?"
print(course_agent.run(f"{SYSTEM_INSTRUCTIONS}\n\nUser query: {test_query}"))

# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

# Part 6: Integrate Your Agent into Our MMAI Agent Discord World (10 points)

You will now integrate an agent into our Discord world. Feel free to use the agent from the previous sections or build an entirely new one. Fun agents are encouraged! After class, we will run all agents at the same time to have them exist together in Discord.

To get started, please join the server using [this link](https://discord.gg/DEzs78ud).

1. Go to https://discord.com/developers/applications/ and click New Application.
2. Open the 'Bot' tab.
3. Set icon (this will be the profile image in Discord) and username.
4. Generate a token and save it. This will be used in the code below.
5. Enable 'Public Bot', 'Presence Intent', 'Server Members Intent', and 'Message Content Intent'.

In [ ]:
!pip install -q -U discord.py
!curl -L "https://github.com/valleballe/mmai/blob/master/static/utils.py?raw=1" -o utils.py

import os
from getpass import getpass

# Set your DISCORD_TOKEN securely in environment variables before running.
if not os.getenv("DISCORD_TOKEN"):
    os.environ["DISCORD_TOKEN"] = getpass("Enter DISCORD_TOKEN: ")

DISCORD_TOKEN = os.getenv("DISCORD_TOKEN")
print("DISCORD_TOKEN loaded from environment.")

Feel free to paste and edit your agent from prior parts of the notebook below.

In [ ]:
# Use your custom course tool from earlier in this notebook.
# This assumes GetCurrentStudentProjectsTool is already defined in Part 3.

from smolagents import CodeAgent, VisitWebpageTool, WebSearchTool
import datetime

# ============================================================
# ######################## CHANGE ME #########################
# ============================================================

SYSTEM_INSTRUCTIONS = (
    'You are a helpful agent for the MIT Modeling Multimodal AI 2026 course. '
    'Cite sources as URLs. If uncertain, say so explicitly.'
    'Course Website: https://mit-mi.github.io/mmai-course/spring2026/\n'
    f'Current date and time is. {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
)

course_agent = CodeAgent(
    tools=[GetCurrentStudentProjectsTool(), VisitWebpageTool(), WebSearchTool()],
    model=model,
)

# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

Next, we will run the agent so that it is accessible on the Discord. You will be able to interact with the agent while the cell below is running. Feel free to play around with what triggers the agent: maybe the agent responds to every single message,  or maybe it only responds when tagged (as in current implementation), or maybe it gets triggered by specific words. Also consider that it could trigger other bots by @tagging them.

In [ ]:
# ============================================================
# ######################## CHANGE ME #########################
# ============================================================

import asyncio
import discord
from discord.ext import commands
from utils import _hydrate_user_mentions


# Discord bot setup
intents = discord.Intents.default()
intents.message_content = True
intents.members = True
bot = commands.Bot(command_prefix="!", intents=intents)

@bot.event
async def on_ready():
    print(f"{bot.user} is online and listening for @mentions.")

@bot.event
async def on_message(message):
    # This triggers whenever someone sends a message in a channel
    if message.author.bot:
        return

    # Check if the agent is mentioned in the message and respond with the agent's answer
    if bot.user and bot.user.mentioned_in(message):
        user_prompt = (
            message.content
            .replace(f"<@{bot.user.id}>", "")
            .replace(f"<@!{bot.user.id}>", "")
            .strip()
        )

        # If the agent is mentioned without a question, prompt the user to ask a question
        if not user_prompt:
            await message.channel.send(
                "Mention me with a question, e.g. @BotName what are current student projects?"
            )
        else:
            full_prompt = f"{SYSTEM_INSTRUCTIONS}\n\nUser question: {user_prompt}"
            async with message.channel.typing():
                response = await asyncio.to_thread(course_agent.run, full_prompt)
                response_text = _hydrate_user_mentions(str(response), message.guild, message.author)

                if len(response_text) > 2000:
                    response_text = response_text[:1997] + "..."

                await message.channel.send(
                    response_text,
                    allowed_mentions=discord.AllowedMentions(users=True, roles=False, everyone=False),
                )

    await bot.process_commands(message)

bot.run(DISCORD_TOKEN)

# ============================================================
# ###################### END CHANGE ME #######################
# ============================================================

To add the agent to the Discord server:
1. Open OAuth2.
2. Enable 'bot' and 'applications.commands'.
3. Under bot permissions, enable 'Send Messages', 'Embed Links', and 'Read message history'. You may need additional permissions depending on your specific needs.

Under 'Generated URL', copy-paste the URL into your browser. This should prompt you to add your agent to a server. Please add it to 'MMAI Agents World'. If you do not see the server, please join it using [this link](https://discord.gg/DEzs78ud).

Reflection and documentation (required):
Write 4-6 sentences reflecting on trigger strategy for your bot. For example, compare always-on response, @mention-only response (this implementation), keyword-triggered response, or letting the LLM decide whether to respond. Include documentation of Discord interactions with the bot in your write-up.

## Optional (10 points): Try OpenClaw

In this optional exercise, experiment with [OpenClaw](https://openclaw.ai/) to explore a more production-style, multi-channel agent system. Set it up locally or via the provided quickstart, connect it to a simple environment (e.g., WhatsApp, Slack, Discord, or CLI), and try building or using at least one “skill” or agent behavior. Submit 2–3 screenshots demonstrating your interaction (e.g., task execution, tool/skill use, or multi-step behavior). In a short reflection (1–2 paragraphs), compare this experience to your smolagents implementation: how does the architecture differ (e.g., abstraction layers, persistence, skills/plugins, orchestration)? Why do you think systems like OpenClaw have become popular? What risks or failure modes emerge when agents are persistent, extensible, and connected to external tools? And lastly, how do you foresee LLM agents developing in the next 5-10 years?